In [ ]:
import numpy as np
import h5py as h5
import matplotlib.pyplot as plt
import os

from scipy import stats


import sys
sys.path.append("../")
from nedflix.conventions import units

In [ ]:
globs = {
    "es": np.logspace(2, 6, 50) * units.GeV,
    "xs": np.linspace(0, np.pi, 400),
    "us": np.linspace(0, 0.9999, 1000), # The slight difference from 0 helps with numerics
    "track_params": (90 * units.GeV, 1.5, 0.5, 1.5), # These parameters were found empirically and are only meant as heuristics
    "cascade_params": (90 * units.GeV, 6, 0.4, 6),
    "fname": "../resources/example_detector_response.h5",
}

In [ ]:
# Make the file if it doesn't exist already
if not os.path.exists(globs["fname"]):
    with h5.File(fname, "w") as _:
        pass

In [ ]:
def shape_parameter(e, e0, a, exp, p0):
    return a  / (np.log10(e) - np.log10(e0))**exp + p0


# Make CDFs
You could get these however you want, but I am just making up something that looks reasonable for now. In practice, you would want to get these from a paper or something

## First we do it for tracks

In [ ]:
plt.plot(globs["es"] / units.GeV, shape_parameter(globs["es"], *globs["track_params"]))
plt.semilogx()
plt.show()

In [ ]:
track_cdfs = np.zeros((len(globs["es"]), len(globs["xs"])))
for idx, e in enumerate(globs["es"]):
    cdf = stats.gamma.cdf(np.degrees(globs["xs"]), shape_parameter(e, *globs["track_params"]))
    track_cdfs[idx, :] = cdf

In [ ]:
# Quick little sanity check
im = plt.imshow(
    track_cdfs.T[::-1],
    aspect="auto", 
    extent=[
        np.log10(globs["es"][0] / units.GeV),
        np.log10(globs["es"][-1] / units.GeV),
        globs["xs"][0],
        globs["xs"][-1]
    ]
)
plt.colorbar(im)
plt.show()

## Now let's do it again for cascades

In [ ]:
plt.plot(globs["es"] / units.GeV, shape_parameter(globs["es"], *globs["cascade_params"]))
plt.semilogx()
plt.show()

In [ ]:
# Put the CDF into a table
cascade_cdfs = np.zeros((len(globs["es"]), len(globs["xs"])))
for idx, e in enumerate(globs["es"]):
    cdf = stats.gamma.cdf(np.degrees(globs["xs"]), shape_parameter(e, *globs["cascade_params"]))
    cascade_cdfs[idx, :] = cdf

In [ ]:
# Plot it just to make sure
im = plt.imshow(
    cascade_cdfs.T[::-1],
    aspect="auto",
    extent=[
        np.log10(globs["es"][0] / units.GeV),
        np.log10(globs["es"][-1] / units.GeV),
        globs["xs"][0],
        globs["xs"][-1]
    ]
)
plt.colorbar(im)
plt.show()

# Okay, now the real fun begins

In [ ]:
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import ridder

In [ ]:
from tqdm import tqdm

In [ ]:
# invert all the CDFS
track_cdfs_interp = RegularGridInterpolator((np.log(globs["es"]), globs["xs"]), track_cdfs)
cascade_cdfs_interp = RegularGridInterpolator((np.log(globs["es"]), globs["xs"]), cascade_cdfs)

inv_track_cdfs = np.zeros((len(globs["es"]), len(globs["us"])))
inv_cascade_cdfs = np.zeros((len(globs["es"]), len(globs["us"])))

for idx, e in enumerate(tqdm(globs["es"])):
    f = lambda x: track_cdfs_interp((np.log(e), x))
    g = lambda x: cascade_cdfs_interp((np.log(e), x))
    for jdx, u in enumerate(globs["us"]):
        f_ = lambda x: f(x) - u
        g_ = lambda x: g(x) - u
        
        inv_track_cdfs[idx, jdx] = ridder(f_, 0, np.pi)
        inv_cascade_cdfs[idx, jdx] = ridder(g_, 0, np.pi)
        

In [ ]:
# plot some of them just to see what up
for alpha, idx in zip([1, 0.8, 0.6, 0.4], [1, 3, 10, 30]):
    plt.plot(globs["us"], inv_track_cdfs[idx, :], color="dodgerblue", alpha=alpha)
    plt.plot(globs["us"], inv_cascade_cdfs[idx, :], color="crimson", alpha=alpha)
plt.show()

In [ ]:
with h5.File(globs["fname"], "r+") as h5f:
    if "cascade_angular_response" in h5f.keys():
        del h5f["cascade_angular_response"]
    h5f.create_group("cascade_angular_response")
    h5f["cascade_angular_response"].create_dataset("us", data=globs["us"])
    h5f["cascade_angular_response"].create_dataset("es", data=globs["es"])
    h5f["cascade_angular_response"].create_dataset("inv_cdfs", data=inv_cascade_cdfs)
    
    if "track_angular_response" in h5f.keys():
        del h5f["track_angular_response"]
    h5f.create_group("track_angular_response")
    h5f["track_angular_response"].create_dataset("us", data=globs["us"])
    h5f["track_angular_response"].create_dataset("es", data=globs["es"])
    h5f["track_angular_response"].create_dataset("inv_cdfs", data=inv_track_cdfs)
    

# Let's look at how to sample this

In [ ]:
# Delete these just to really play pretend
try:
    del inv_track_cdfs
except NameError:
    pass
try:
    del track_cdfs
except NameError:
    pass
try:
    del track_cdfs_interp
except NameError:
    pass
try:
    del inv_cascade_cdfs
except NameError:
    pass
try:
    del cascade_cdfs
except NameError:
    pass
try:
    del cascade_cdfs_interp
except NameError:
    pass

In [ ]:
# First we interpolate
with h5.File(globs["fname"]) as h5f:
    cascade_i = RegularGridInterpolator(
        (np.log(h5f["cascade_angular_response/es"][:]), h5f["cascade_angular_response/us"][:]),
        h5f["cascade_angular_response/inv_cdfs"][:]
    )
    track_i = RegularGridInterpolator(
        (np.log(h5f["track_angular_response/es"][:]), h5f["track_angular_response/us"][:]),
        h5f["track_angular_response/inv_cdfs"][:]
    )

In [ ]:
def sample_2d(e, interp, n=1):
    u = np.random.rand(n)
    # Replace values with those above the maximum u value.
    if hasattr(u, "__iter__"):
        u = np.where(u > interp.grid[1].max(), interp.grid[1].max(), u)
        u = np.where(u < interp.grid[1].min(), interp.grid[1].min(), u)
    return interp((np.log(e), u))

In [ ]:
# Plot it on a fine grid to make sure that the interpolating hasn't done anything bananas
es_fine = np.logspace(2, 6, 1000) * units.GeV
track_means = []
cascade_means = []
for e in tqdm(es_fine):
    track_means.append(np.mean(sample_2d(e, track_i, 10_000)))
    cascade_means.append(np.mean(sample_2d(e, cascade_i, 10_000)))
    
plt.plot(globs["es"], shape_parameter(globs["es"], *globs["cascade_params"]), lw=4, ls="--")
plt.plot(globs["es"], shape_parameter(globs["es"], *globs["track_params"]), lw=4, ls="--")
plt.plot(es_fine, np.degrees(cascade_means))
plt.plot(es_fine, np.degrees(track_means))
plt.semilogx()
plt.show()

In [ ]:
# # This is slow, but easily implemented. You can look at it for your own edification if you would like
# def sample_misreco_angle(e:float, interp: RegularGridInterpolator, n: int=1) -> float:
#     if n==1:
#         u = np.random.rand()
#         f = lambda x: interp((np.log(e), x)) - u
#         return ridder(f, 0, np.pi)
#     else:
#         us = np.random.rand(n)
#         out = np.zeros(us.shape)
#         for idx, u in enumerate(us):
#             f = lambda x: interp((np.log(e), x)) - u
#             out[idx] = ridder(f, 0, np.pi)
#         return out

# Now let's do this for energy.
This will be much simpler

## Make some fake data

In [ ]:
sigma_cascade = (np.log(1.15) - np.log(0.85)) / 2
sigma_track = (np.log(3) - np.log(0.3)) / 2

In [ ]:
xs_track = np.linspace(-3 * sigma_track, 3*sigma_track, 2000)
xs_cascade = np.linspace(-3 * sigma_cascade, 3*sigma_cascade, 2000)
pdf_track = stats.norm.pdf(xs_track, loc=0, scale=sigma_track)
pdf_cascade = stats.norm.pdf(xs_cascade, loc=0, scale=sigma_cascade)

# Make these into CDFs
cdf_track = np.cumsum(pdf_track) / np.sum(pdf_track)
cdf_cascade = np.cumsum(pdf_cascade) / np.sum(pdf_cascade)

# Plot just to make sure the bootleg CDF calculation matches the analytic expectation
plt.plot(xs_track, stats.norm.cdf(xs_track, loc=0, scale=sigma_track), lw=4, ls="--")
plt.plot(xs_track, cdf_track)
plt.plot(xs_cascade, stats.norm.cdf(xs_cascade, loc=0, scale=sigma_cascade), lw=4, ls="--")
plt.plot(xs_cascade, cdf_cascade)
plt.show()

In [ ]:
from scipy.interpolate import interp1d

In [ ]:
cdf_interp_track = interp1d(xs_track, cdf_track)
cdf_interp_cascade = interp1d(xs_cascade, cdf_cascade)

us = np.linspace(0, 1, 10000)
track_inv_cdf = np.zeros(us.shape)
cascade_inv_cdf = np.zeros(us.shape)

for idx, u in enumerate(tqdm(us)):
    if u==0:
        track_inv_cdf[idx] = xs_track.min()
        cascade_inv_cdf[idx] = xs_cascade.min()
        continue
    if u==1:
        track_inv_cdf[idx] = xs_track.max()
        cascade_inv_cdf[idx] = xs_cascade.max()
        continue
    f = lambda x: stats.norm.cdf(x, loc=0, scale=sigma_track) - u
    g = lambda x: stats.norm.cdf(x, loc=0, scale=sigma_cascade) - u
    track_inv_cdf[idx] = ridder(f, -5*sigma_track, 5*sigma_track)
    cascade_inv_cdf[idx] = ridder(g, -5*sigma_cascade, 5*sigma_cascade)

In [ ]:
# Quick check :-)
plt.plot(cdf_cascade, xs_cascade, lw=2, ls="--")
plt.plot(us, cascade_inv_cdf, label="cascade")
plt.plot(cdf_track, xs_track, lw=2, ls="--")
plt.plot(us, track_inv_cdf, label="track")
plt.legend()
plt.show()

In [ ]:
with h5.File(globs["fname"], "r+") as h5f:
    if "cascade_energy_resolution" in h5f.keys():
        del h5f["cascade_energy_resolution"]
    h5f.create_group("cascade_energy_resolution")
    h5f["cascade_energy_resolution"].create_dataset("us", data=us)
    h5f["cascade_energy_resolution"].create_dataset("inv_cdf", data=cascade_inv_cdf)

    if "track_energy_resolution" in h5f.keys():
        del h5f["track_energy_resolution"]
    h5f.create_group("track_energy_resolution")
    h5f["track_energy_resolution"].create_dataset("us", data=us)
    h5f["track_energy_resolution"].create_dataset("inv_cdf", data=track_inv_cdf)

## And once again, how we sample it

In [ ]:
try:
    del cascade_inv_cdf
except NameError:
    pass
try:
    del track_inv_cdf
except NameError:
    pass
try:
    del cdf_interp_cascade
except NameError:
    pass
try:
    del cdf_interp_track
except NameError:
    pass
try:
    del cdf_cascade
except NameError:
    pass
try:
    del cdf_track
except NameError:
    pass

In [ ]:
with h5.File(globs["fname"]) as h5f:
    cascade_i = interp1d(
        h5f["cascade_energy_resolution/us"][:],
        h5f["cascade_energy_resolution/inv_cdf"][:]
    )
    track_i = interp1d(
        h5f["track_energy_resolution/us"][:],
        h5f["track_energy_resolution/inv_cdf"][:]
    )

In [ ]:
def sample_energy(interp, n=1):
    u = np.random.rand(n)
    return np.exp(interp(u))

In [ ]:
np.median(sample_energy(track_i, 100_000))

In [ ]:
h, bins = np.histogram(np.log10(sample_energy(track_i, 100_000)), bins=11)
cents = (bins[1:] + bins[:-1]) / 2
plt.step(cents, h, where="mid")

h, bins = np.histogram(np.log10(sample_energy(cascade_i, 100_000)), bins=11)
cents = (bins[1:] + bins[:-1]) / 2
plt.step(cents, h, where="mid")

plt.show()

## Finally let's do the effective area
I think this is gonna be the most "artful"

In [ ]:
# Load up tabulated effective from the data release
a = np.genfromtxt("../resources/IC86_II_effectiveArea.csv")

In [ ]:
dec_cents = (np.radians(np.unique(a[:,2])) + np.radians(np.unique(a[:,3]))) / 2
e_cents = (np.power(10, np.unique(a[:, 0])) + np.power(10, np.unique(a[:, 1]))) * units.GeV/ 2

In [ ]:
# Plop all the effective areas into an array
effa = np.zeros((len(e_cents), len(dec_cents)))
for row in tqdm(a): 
    e = np.mean(np.power(10, row[[0, 1]])) * units.GeV
    dec = (np.radians(row[2]) + np.radians(row[3])) / 2
    idx = np.argmin(np.abs(e - e_cents))
    jdx = np.argmin(np.abs(dec - dec_cents))
    effa[idx, jdx] = row[4] * units.cm**2

In [ ]:
# Plot the data just to see what it looks like
fig, ax = plt.subplots()

im = ax.imshow(
    np.log10(effa / units.cm**2)[::-1, :],
    aspect="auto",
    extent=[
        -1, 1,
#         np.sin(a[:, [2,3]]).min(),
#         np.sin(a[:, [2,3]]).max(),
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    zorder=1,
    vmin=-2,
    vmax=8.1
)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 10)

ax.set_yticks(range(2,11))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 11)])

cbar = plt.colorbar(im, label=r"$\log_{10}\left(A_{\mathrm{eff}} / \mathrm{cm}^{2}\right)$")

plt.show()

This spotty pocket in the bottom left corner is worrying. I've tried to come up with some clever ways to deal with it, but ultimately, I decided we are just going to ignore it for now.

In [ ]:
# Let's take a look at the proposed cuts
lower_bounds = [
    [np.log10(2.5e4 * units.GeV)],
    [-15.9862667245, np.log10(np.power(10, -0.397940008675) * units.GeV)],
]
upper_bounds = [
    [np.log10(1e9 * units.GeV)]
]

fig, ax = plt.subplots()

im = ax.imshow(
    np.log10(effa / units.cm**2)[::-1, :],
    aspect="auto",
    extent=[
        -1, 1,
#         np.sin(a[:, [2,3]]).min(),
#         np.sin(a[:, [2,3]]).max(),
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    vmin=-2,
    vmax=8.1
)

plt.plot(
    [-1, -0.3],
    np.poly1d(lower_bounds[0])([-1, -0.3]) - np.log10(units.GeV),
    lw=4,
    c="crimson"
)
plt.plot(
    [-0.3, -0.15],
    np.poly1d(lower_bounds[1])([-0.3, -0.15]) - np.log10(units.GeV),
    lw=4,
    c="crimson"
)
plt.plot(
    [-1, 1],
    np.poly1d(upper_bounds[0])([-1, 1]) - np.log10(units.GeV),
    lw=4,
    c="crimson"
)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 10)

ax.set_yticks(range(2,11))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 11)])

cbar = plt.colorbar(im, label=r"$\log_{10}\left(A_{\mathrm{eff}} / \mathrm{cm}^{2}\right)$")

plt.show()

In [ ]:
def poly_bounds(sindec, e, lower_bounds, upper_bounds):
    
    y = np.log10(e)
    
    upper_bool = False
    for bound in upper_bounds:
        f = np.poly1d(bound)
        if y < f(sindec):
            upper_bool = True
            break
    
    lower_bool = False
    for bound in lower_bounds:
        f = np.poly1d(bound)
        if y > f(sindec):
            lower_bool = True
    
    return lower_bool and upper_bool

In [ ]:
out = np.zeros((len(dec_cents), len(e_cents)), dtype=bool)
for idx, dec in enumerate(dec_cents):
    for jdx, e in enumerate(e_cents):
        out[idx, jdx] = poly_bounds(np.sin(dec), e, lower_bounds, upper_bounds)

out = out[:, ::-1].T
        
fig, ax = plt.subplots() 

im = ax.imshow(
    out,
    aspect="auto",
    extent=[-1, 1, a[:, [0, 1]].min(), a[:, [0, 1]].max()],
    cmap="Greys_r"
)

cbar = plt.colorbar(im)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 10)

ax.set_yticks(range(2, 10))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 10)])

plt.show()

In [ ]:
fig, ax = plt.subplots()

for b in np.linspace(-10, 20, 150):
    s = 4.5
    fp = lambda x: s * x + b
    fn = lambda x: -s * x + b
    xs_ = np.linspace(-1, 1, 3)
    ax.plot(xs_, fp(xs_), c="k", zorder=0, lw=0.2, alpha=0.5)
    ax.plot(xs_, fn(xs_), c="k", zorder=0, lw=0.2, alpha=0.5)
    del xs_

im = ax.imshow(
    np.where(out, np.log10(effa / units.cm**2)[::-1, :], -np.inf),
    aspect="auto",
    extent=[
        -1, 1,
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    zorder=1,
    vmin=-2,
    vmax=8.1
    
)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 10)

ax.set_yticks(range(2,10))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 10)])

cbar = plt.colorbar(im, label=r"$\log_{10}\left(A_{\mathrm{eff}} / \mathrm{cm}^{2}\right)$")

plt.show()

In [ ]:
def harmonize_bounds(bounds):
    max_length = 0
    for bound in bounds:
        l = 0
        while l<len(bound) and bound[l]==0:
            l += 1
        max_length = max(max_length, len(bound)-l)
    harmonized_bounds = np.zeros((len(bounds), max_length))
    for idx, bound in enumerate(bounds):
        bd = bound.copy()
        d = max_length - len(bd)
        if d > 0:
            bd = np.append(np.zeros(d), bd)
        harmonized_bounds[idx, :] = bd
    return harmonized_bounds

In [ ]:
with h5.File(globs["fname"], "r+") as h5f:
    k = "track_effective_area"
    if k in h5f.keys():
        del h5f[k]
    h5f.create_group(k)
    h5f[k].create_dataset("declinations", data=dec_cents)
    h5f[k].create_dataset("es", data=e_cents)
    h5f[k].create_dataset("tabulated_values", data=effa)
    h5f[k].create_dataset("lower_bounds", data=harmonize_bounds(lower_bounds))
    h5f[k].create_dataset("upper_bounds", data=harmonize_bounds(upper_bounds))
    
    # For now, we're going to use the same effective area twice, but in practice this is different
    k = "cascade_effective_area"
    if k in h5f.keys():
        del h5f[k]
    h5f.create_group(k)
    h5f[k].create_dataset("declinations", data=dec_cents)
    h5f[k].create_dataset("es", data=e_cents)
    h5f[k].create_dataset("tabulated_values", data=effa)
    h5f[k].create_dataset("lower_bounds", data=harmonize_bounds(lower_bounds))
    h5f[k].create_dataset("upper_bounds", data=harmonize_bounds(upper_bounds))

## Finally let's use this

In [ ]:
def effa_helper(decs, es, tabulated_values, lower_bounds, upper_bounds):
    sin_decs = np.sin(decs)
    tabulated_values_scrubbed = np.where(
        tabulated_values>0,
        tabulated_values,
        tabulated_values[tabulated_values>0].min()
    )
    i = RegularGridInterpolator((np.log(es), sin_decs), np.log(tabulated_values_scrubbed))

    def f(dec, e, interp, lbs, ubs):
        sd = np.sin(dec)
        le = np.log(e)
        # return 0 if outside of interpolator bounds
        if le < interp.grid[0][0] or le > interp.grid[0][-1]:
            return 0.0
        if sd < interp.grid[1][0] or sd > interp.grid[1][-1]:
            return 0.0
        # return 0 if bounds say so
        if not poly_bounds(sd, e, lower_bounds, upper_bounds):
            return 0.0
        return np.exp(interp((le, sd)))
    
    return lambda dec, e: f(dec, e, i, lower_bounds, upper_bounds)

In [ ]:
with h5.File(globs["fname"]) as h5f:
    k = "track_effective_area"

    decs = h5f[f"{k}/declinations"]
    es = h5f[f"{k}/es"]
    tabulated_values = h5f[f"{k}/tabulated_values"]
    lower_bounds = h5f[f"{k}/lower_bounds"]
    upper_bounds = h5f[f"{k}/upper_bounds"]
    effa_f = effa_helper(decs[:], es[:], tabulated_values[:], lower_bounds[:], upper_bounds[:])

In [ ]:
# Make sure this works
es_fine = np.logspace(np.log10(e_cents.min()), np.log10(e_cents.max()), 200)
decs_fine = np.linspace(-1, 1, 200)
effa_new = np.zeros(es_fine.shape + decs_fine.shape)
for idx, e in enumerate(es_fine):
    for jdx, dec in enumerate(decs_fine):
        effa_new[idx, jdx] = effa_f(dec, e)
        
fig, ax = plt.subplots()

for b in np.linspace(-10, 20, 150):
    s = 4.5
    fp = lambda x: s * x + b
    fn = lambda x: -s * x + b
    xs_ = np.linspace(-1, 1, 3)
    ax.plot(xs_, fp(xs_), c="k", zorder=0, lw=0.2, alpha=0.5)
    ax.plot(xs_, fn(xs_), c="k", zorder=0, lw=0.2, alpha=0.5)
    del xs_

im = ax.imshow(
    np.log10(effa_new / units.cm**2)[::-1, :],
    aspect="auto",
    extent=[
        -1, 1,
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    zorder=1,
    vmin=-2,
    vmax=8.1
)

cbar = plt.colorbar(im)

plt.show()

# Detritis

In [ ]:
fig, ax = plt.subplots()

im = ax.imshow(
    np.log10(effa)[::-1, :],
    aspect="auto",
    extent=[
        -1, 1,
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    vmin = -2,
    vmax = 8
)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 9)

ax.set_yticks(range(2,10))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 10)])

cbar = plt.colorbar(im)

plt.show()

#

fig, ax = plt.subplots()

im = ax.imshow(
    np.log10(new_effa)[::-1, :],
    aspect="auto",
    extent=[
        -1, 1,
        a[:, [0, 1]].min(),
        a[:, [0, 1]].max(),
    ],
    vmin = -2,
    vmax = 8
)

ax.set_xlabel(r"$\sin\left(\delta\right)$")
ax.set_ylabel(r"$E_{\nu}~\left[\mathrm{GeV}\right]$")

ax.set_ylim(2, 9)

ax.set_yticks(range(2,10))
ax.set_yticklabels([r"$10^{%d}$" % x for x in range(2, 10)])

cbar = plt.colorbar(im)

plt.show()

In [ ]:
for idx in range(50):
    mask = slice(idx * 40, (idx+1) * 40)
    es = np.mean(np.power(10, a[mask, [0, 1]]), axis=1)
    plt.step(es, effa[:, idx], where="mid")
    plt.loglog()
    plt.show()

In [ ]:
for idx in range(50):
    mask = slice(idx * 40, (idx+1) * 40)
    es = np.mean(np.power(10, a[mask, [0, 1]]), axis=1)
    plt.step(es, new_effa[:, idx], where="mid")
    plt.loglog()
    plt.show()